# T5/T8 DGX batch
Pin the published deployment commit with MOM_BPE_DEPLOY_REVISION. Each cell starts a fresh process and checks its explicit prerequisites. The four reviewed jobs remain disabled until authorized.


## 1. Verify the pinned Python environment


In [ ]:
import os
import subprocess

python = "/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python"
probe = """
import importlib.metadata as md
import platform
import unicodedata
import torch

versions = {"regex": "2026.9.29", "tokenizers": "0.23.3", "PyYAML": "6.0.3", "fasttext-wheel": "0.9.2",
            "pyarrow": "26.0.0", "numpy": "2.5.2", "torch": "2.5.1+cu121"}
observed = {name: md.version(name) for name in versions}
runtime = (platform.system(), platform.python_version(), unicodedata.unidata_version, torch.version.cuda)
if runtime != ("Linux", "3.12.0", "15.0.0", "12.1") or observed != versions:
    raise RuntimeError(f"DGX environment differs: {runtime}, {observed}")
md.version("huggingface_hub")
print("Verified the pinned DGX Python environment and Hugging Face client")
"""
env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_OFFLINE="1", HF_DATASETS_OFFLINE="1")
subprocess.run([python, "-I", "-B", "-c", probe], env=env, check=True)


## 2. Retrieve exact GitHub source commits


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch("[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)
if source.resolve() != source or deployment.resolve() != deployment:
    raise RuntimeError("Unexpected source symlink")

import fcntl
from tempfile import mkdtemp

repository = "https://github.com/Akshai0007/Thesi_Tokenizer_Pipeline.git"
env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_LFS_SKIP_SMUDGE="1")
env.pop("HF_TOKEN", None)
env.pop("HUGGING_FACE_HUB_TOKEN", None)

def git(directory, *args):
    return subprocess.check_output(["git", "-C", str(directory), *args], env=env, text=True).strip()

with (root / ".t5-t8-source.lock").open("a+b") as lock:
    fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    for directory, pin in ((deployment, revision), (source, operational)):
        if not directory.exists():
            staging = Path(mkdtemp(prefix="mom-source-", dir=root))
            git(staging, "init", "--quiet")
            git(staging, "fetch", "--no-tags", "--depth=1", repository, pin)
            git(staging, "checkout", "--detach", "--quiet", "FETCH_HEAD")
            if git(staging, "rev-parse", "HEAD") != pin or directory.exists():
                raise RuntimeError("GitHub checkout revision or destination conflict")
            staging.rename(directory)
        if git(directory, "rev-parse", "HEAD") != pin:
            raise RuntimeError(f"GitHub commit mismatch at {directory.name}")
print("Retrieved both pinned GitHub checkouts")


## 3. Verify the notebook and deployment file hashes


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch("[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)
if source.resolve() != source or deployment.resolve() != deployment:
    raise RuntimeError("Unexpected source symlink")

def run(action, *options):
    directory = deployment / "experiments/mom_bpe"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_DISABLE_TELEMETRY="1",
               HF_HUB_OFFLINE="0" if action in ("authenticate", "restore-individual", "backup-hub") else "1",
               HF_DATASETS_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    env.pop("MLTOK_PLUGINS", None)
    head = subprocess.check_output(["git", "-C", str(deployment), "rev-parse", "HEAD"], text=True).strip()
    if head != revision:
        raise RuntimeError("GitHub deployment commit mismatch")
    for filename in ("dgx_batch.py", "t5_t8_dgx.py"):
        blob = subprocess.check_output(["git", "-C", str(deployment), "show", f"{revision}:experiments/mom_bpe/{filename}"])
        path = directory / filename
        if path.is_symlink() or not path.is_file() or path.read_bytes() != blob:
            raise RuntimeError(f"Missing or inconsistent GitHub source {filename}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(python), "-I", "-B", str(directory / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--deployment-revision", revision,
                    "--session", session, *options], cwd=source, env=env, check=True)

run("source")
run("environment")


## 4. Authenticate to the pinned private Hugging Face revision


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch("[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)
if source.resolve() != source or deployment.resolve() != deployment:
    raise RuntimeError("Unexpected source symlink")

def run(action, *options):
    directory = deployment / "experiments/mom_bpe"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_DISABLE_TELEMETRY="1",
               HF_HUB_OFFLINE="0" if action in ("authenticate", "restore-individual", "backup-hub") else "1",
               HF_DATASETS_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    env.pop("MLTOK_PLUGINS", None)
    head = subprocess.check_output(["git", "-C", str(deployment), "rev-parse", "HEAD"], text=True).strip()
    if head != revision:
        raise RuntimeError("GitHub deployment commit mismatch")
    for filename in ("dgx_batch.py", "t5_t8_dgx.py"):
        blob = subprocess.check_output(["git", "-C", str(deployment), "show", f"{revision}:experiments/mom_bpe/{filename}"])
        path = directory / filename
        if path.is_symlink() or not path.is_file() or path.read_bytes() != blob:
            raise RuntimeError(f"Missing or inconsistent GitHub source {filename}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(python), "-I", "-B", str(directory / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--deployment-revision", revision,
                    "--session", session, *options], cwd=source, env=env, check=True)


from getpass import getpass

if not os.environ.get("HF_TOKEN"):
    token = getpass("Hugging Face read token (blank uses approved cached credentials): ")
    if token:
        os.environ["HF_TOKEN"] = token
    del token
run("authenticate")


## 5. Restore individual artifacts and display missing sources


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch("[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)
if source.resolve() != source or deployment.resolve() != deployment:
    raise RuntimeError("Unexpected source symlink")

def run(action, *options):
    directory = deployment / "experiments/mom_bpe"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_DISABLE_TELEMETRY="1",
               HF_HUB_OFFLINE="0" if action in ("authenticate", "restore-individual", "backup-hub") else "1",
               HF_DATASETS_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    env.pop("MLTOK_PLUGINS", None)
    head = subprocess.check_output(["git", "-C", str(deployment), "rev-parse", "HEAD"], text=True).strip()
    if head != revision:
        raise RuntimeError("GitHub deployment commit mismatch")
    for filename in ("dgx_batch.py", "t5_t8_dgx.py"):
        blob = subprocess.check_output(["git", "-C", str(deployment), "show", f"{revision}:experiments/mom_bpe/{filename}"])
        path = directory / filename
        if path.is_symlink() or not path.is_file() or path.read_bytes() != blob:
            raise RuntimeError(f"Missing or inconsistent GitHub source {filename}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(python), "-I", "-B", str(directory / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--deployment-revision", revision,
                    "--session", session, *options], cwd=source, env=env, check=True)

run("restore-individual")


## 6. Verify all frozen artifact hashes


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch("[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)
if source.resolve() != source or deployment.resolve() != deployment:
    raise RuntimeError("Unexpected source symlink")

def run(action, *options):
    directory = deployment / "experiments/mom_bpe"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_DISABLE_TELEMETRY="1",
               HF_HUB_OFFLINE="0" if action in ("authenticate", "restore-individual", "backup-hub") else "1",
               HF_DATASETS_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    env.pop("MLTOK_PLUGINS", None)
    head = subprocess.check_output(["git", "-C", str(deployment), "rev-parse", "HEAD"], text=True).strip()
    if head != revision:
        raise RuntimeError("GitHub deployment commit mismatch")
    for filename in ("dgx_batch.py", "t5_t8_dgx.py"):
        blob = subprocess.check_output(["git", "-C", str(deployment), "show", f"{revision}:experiments/mom_bpe/{filename}"])
        path = directory / filename
        if path.is_symlink() or not path.is_file() or path.read_bytes() != blob:
            raise RuntimeError(f"Missing or inconsistent GitHub source {filename}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(python), "-I", "-B", str(directory / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--deployment-revision", revision,
                    "--session", session, *options], cwd=source, env=env, check=True)

run("verify")


## 7. Verify the four exposure plans


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch("[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)
if source.resolve() != source or deployment.resolve() != deployment:
    raise RuntimeError("Unexpected source symlink")

def run(action, *options):
    directory = deployment / "experiments/mom_bpe"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_DISABLE_TELEMETRY="1",
               HF_HUB_OFFLINE="0" if action in ("authenticate", "restore-individual", "backup-hub") else "1",
               HF_DATASETS_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    env.pop("MLTOK_PLUGINS", None)
    head = subprocess.check_output(["git", "-C", str(deployment), "rev-parse", "HEAD"], text=True).strip()
    if head != revision:
        raise RuntimeError("GitHub deployment commit mismatch")
    for filename in ("dgx_batch.py", "t5_t8_dgx.py"):
        blob = subprocess.check_output(["git", "-C", str(deployment), "show", f"{revision}:experiments/mom_bpe/{filename}"])
        path = directory / filename
        if path.is_symlink() or not path.is_file() or path.read_bytes() != blob:
            raise RuntimeError(f"Missing or inconsistent GitHub source {filename}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(python), "-I", "-B", str(directory / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--deployment-revision", revision,
                    "--session", session, *options], cwd=source, env=env, check=True)

run("plan")


## 8. Check CPU readiness and private scratch


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch("[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)
if source.resolve() != source or deployment.resolve() != deployment:
    raise RuntimeError("Unexpected source symlink")

def run(action, *options):
    directory = deployment / "experiments/mom_bpe"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_DISABLE_TELEMETRY="1",
               HF_HUB_OFFLINE="0" if action in ("authenticate", "restore-individual", "backup-hub") else "1",
               HF_DATASETS_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    env.pop("MLTOK_PLUGINS", None)
    head = subprocess.check_output(["git", "-C", str(deployment), "rev-parse", "HEAD"], text=True).strip()
    if head != revision:
        raise RuntimeError("GitHub deployment commit mismatch")
    for filename in ("dgx_batch.py", "t5_t8_dgx.py"):
        blob = subprocess.check_output(["git", "-C", str(deployment), "show", f"{revision}:experiments/mom_bpe/{filename}"])
        path = directory / filename
        if path.is_symlink() or not path.is_file() or path.read_bytes() != blob:
            raise RuntimeError(f"Missing or inconsistent GitHub source {filename}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(python), "-I", "-B", str(directory / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--deployment-revision", revision,
                    "--session", session, *options], cwd=source, env=env, check=True)

run("ready", "--scope", "all")


## 9. Inspect GPU and scratch capacity


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch("[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)
if source.resolve() != source or deployment.resolve() != deployment:
    raise RuntimeError("Unexpected source symlink")

def run(action, *options):
    directory = deployment / "experiments/mom_bpe"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_DISABLE_TELEMETRY="1",
               HF_HUB_OFFLINE="0" if action in ("authenticate", "restore-individual", "backup-hub") else "1",
               HF_DATASETS_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    env.pop("MLTOK_PLUGINS", None)
    head = subprocess.check_output(["git", "-C", str(deployment), "rev-parse", "HEAD"], text=True).strip()
    if head != revision:
        raise RuntimeError("GitHub deployment commit mismatch")
    for filename in ("dgx_batch.py", "t5_t8_dgx.py"):
        blob = subprocess.check_output(["git", "-C", str(deployment), "show", f"{revision}:experiments/mom_bpe/{filename}"])
        path = directory / filename
        if path.is_symlink() or not path.is_file() or path.read_bytes() != blob:
            raise RuntimeError(f"Missing or inconsistent GitHub source {filename}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(python), "-I", "-B", str(directory / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--deployment-revision", revision,
                    "--session", session, *options], cwd=source, env=env, check=True)

run("inspect")


## 10. Run an approved A100 diagnostic


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch("[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)
if source.resolve() != source or deployment.resolve() != deployment:
    raise RuntimeError("Unexpected source symlink")

def run(action, *options):
    directory = deployment / "experiments/mom_bpe"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_DISABLE_TELEMETRY="1",
               HF_HUB_OFFLINE="0" if action in ("authenticate", "restore-individual", "backup-hub") else "1",
               HF_DATASETS_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    env.pop("MLTOK_PLUGINS", None)
    head = subprocess.check_output(["git", "-C", str(deployment), "rev-parse", "HEAD"], text=True).strip()
    if head != revision:
        raise RuntimeError("GitHub deployment commit mismatch")
    for filename in ("dgx_batch.py", "t5_t8_dgx.py"):
        blob = subprocess.check_output(["git", "-C", str(deployment), "show", f"{revision}:experiments/mom_bpe/{filename}"])
        path = directory / filename
        if path.is_symlink() or not path.is_file() or path.read_bytes() != blob:
            raise RuntimeError(f"Missing or inconsistent GitHub source {filename}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(python), "-I", "-B", str(directory / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--deployment-revision", revision,
                    "--session", session, *options], cwd=source, env=env, check=True)


smoke_approved = False
allocated = False
shared_approved = False
uuid = ""
scope = "all"
if smoke_approved and allocated:
    sharing = ["--shared"] if shared_approved else []
    run("smoke", "--enable", "--allocated", "--uuid", uuid, "--scope", scope, *sharing)
else:
    print("A100 diagnostic disabled")


## 11. Execute the first matched pair


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch("[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)
if source.resolve() != source or deployment.resolve() != deployment:
    raise RuntimeError("Unexpected source symlink")

def run(action, *options):
    directory = deployment / "experiments/mom_bpe"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_DISABLE_TELEMETRY="1",
               HF_HUB_OFFLINE="0" if action in ("authenticate", "restore-individual", "backup-hub") else "1",
               HF_DATASETS_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    env.pop("MLTOK_PLUGINS", None)
    head = subprocess.check_output(["git", "-C", str(deployment), "rev-parse", "HEAD"], text=True).strip()
    if head != revision:
        raise RuntimeError("GitHub deployment commit mismatch")
    for filename in ("dgx_batch.py", "t5_t8_dgx.py"):
        blob = subprocess.check_output(["git", "-C", str(deployment), "show", f"{revision}:experiments/mom_bpe/{filename}"])
        path = directory / filename
        if path.is_symlink() or not path.is_file() or path.read_bytes() != blob:
            raise RuntimeError(f"Missing or inconsistent GitHub source {filename}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(python), "-I", "-B", str(directory / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--deployment-revision", revision,
                    "--session", session, *options], cwd=source, env=env, check=True)


enable_training = False
if enable_training:
    run("first", "--enable")
else:
    print("GPU training disabled")


## 12. Verify the first pair and write its comparison


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch("[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)
if source.resolve() != source or deployment.resolve() != deployment:
    raise RuntimeError("Unexpected source symlink")

def run(action, *options):
    directory = deployment / "experiments/mom_bpe"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_DISABLE_TELEMETRY="1",
               HF_HUB_OFFLINE="0" if action in ("authenticate", "restore-individual", "backup-hub") else "1",
               HF_DATASETS_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    env.pop("MLTOK_PLUGINS", None)
    head = subprocess.check_output(["git", "-C", str(deployment), "rev-parse", "HEAD"], text=True).strip()
    if head != revision:
        raise RuntimeError("GitHub deployment commit mismatch")
    for filename in ("dgx_batch.py", "t5_t8_dgx.py"):
        blob = subprocess.check_output(["git", "-C", str(deployment), "show", f"{revision}:experiments/mom_bpe/{filename}"])
        path = directory / filename
        if path.is_symlink() or not path.is_file() or path.read_bytes() != blob:
            raise RuntimeError(f"Missing or inconsistent GitHub source {filename}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(python), "-I", "-B", str(directory / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--deployment-revision", revision,
                    "--session", session, *options], cwd=source, env=env, check=True)

run("results", "--scope", "first")
run("analyze")


## 13. Execute the second matched pair


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch("[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)
if source.resolve() != source or deployment.resolve() != deployment:
    raise RuntimeError("Unexpected source symlink")

def run(action, *options):
    directory = deployment / "experiments/mom_bpe"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_DISABLE_TELEMETRY="1",
               HF_HUB_OFFLINE="0" if action in ("authenticate", "restore-individual", "backup-hub") else "1",
               HF_DATASETS_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    env.pop("MLTOK_PLUGINS", None)
    head = subprocess.check_output(["git", "-C", str(deployment), "rev-parse", "HEAD"], text=True).strip()
    if head != revision:
        raise RuntimeError("GitHub deployment commit mismatch")
    for filename in ("dgx_batch.py", "t5_t8_dgx.py"):
        blob = subprocess.check_output(["git", "-C", str(deployment), "show", f"{revision}:experiments/mom_bpe/{filename}"])
        path = directory / filename
        if path.is_symlink() or not path.is_file() or path.read_bytes() != blob:
            raise RuntimeError(f"Missing or inconsistent GitHub source {filename}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(python), "-I", "-B", str(directory / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--deployment-revision", revision,
                    "--session", session, *options], cwd=source, env=env, check=True)


enable_training = False
if enable_training:
    run("second", "--enable")
else:
    print("GPU training disabled")


## 14. Verify all four results and final checkpoints


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch("[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)
if source.resolve() != source or deployment.resolve() != deployment:
    raise RuntimeError("Unexpected source symlink")

def run(action, *options):
    directory = deployment / "experiments/mom_bpe"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_DISABLE_TELEMETRY="1",
               HF_HUB_OFFLINE="0" if action in ("authenticate", "restore-individual", "backup-hub") else "1",
               HF_DATASETS_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    env.pop("MLTOK_PLUGINS", None)
    head = subprocess.check_output(["git", "-C", str(deployment), "rev-parse", "HEAD"], text=True).strip()
    if head != revision:
        raise RuntimeError("GitHub deployment commit mismatch")
    for filename in ("dgx_batch.py", "t5_t8_dgx.py"):
        blob = subprocess.check_output(["git", "-C", str(deployment), "show", f"{revision}:experiments/mom_bpe/{filename}"])
        path = directory / filename
        if path.is_symlink() or not path.is_file() or path.read_bytes() != blob:
            raise RuntimeError(f"Missing or inconsistent GitHub source {filename}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(python), "-I", "-B", str(directory / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--deployment-revision", revision,
                    "--session", session, *options], cwd=source, env=env, check=True)

run("results", "--scope", "all")


## 15. Generate paired development comparisons


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch("[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)
if source.resolve() != source or deployment.resolve() != deployment:
    raise RuntimeError("Unexpected source symlink")

def run(action, *options):
    directory = deployment / "experiments/mom_bpe"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_DISABLE_TELEMETRY="1",
               HF_HUB_OFFLINE="0" if action in ("authenticate", "restore-individual", "backup-hub") else "1",
               HF_DATASETS_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    env.pop("MLTOK_PLUGINS", None)
    head = subprocess.check_output(["git", "-C", str(deployment), "rev-parse", "HEAD"], text=True).strip()
    if head != revision:
        raise RuntimeError("GitHub deployment commit mismatch")
    for filename in ("dgx_batch.py", "t5_t8_dgx.py"):
        blob = subprocess.check_output(["git", "-C", str(deployment), "show", f"{revision}:experiments/mom_bpe/{filename}"])
        path = directory / filename
        if path.is_symlink() or not path.is_file() or path.read_bytes() != blob:
            raise RuntimeError(f"Missing or inconsistent GitHub source {filename}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(python), "-I", "-B", str(directory / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--deployment-revision", revision,
                    "--session", session, *options], cwd=source, env=env, check=True)

run("analyze")


## 16. Optionally back up verified outputs to a private Hugging Face repository


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch("[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)
if source.resolve() != source or deployment.resolve() != deployment:
    raise RuntimeError("Unexpected source symlink")

def run(action, *options):
    directory = deployment / "experiments/mom_bpe"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_DISABLE_TELEMETRY="1",
               HF_HUB_OFFLINE="0" if action in ("authenticate", "restore-individual", "backup-hub") else "1",
               HF_DATASETS_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    env.pop("MLTOK_PLUGINS", None)
    head = subprocess.check_output(["git", "-C", str(deployment), "rev-parse", "HEAD"], text=True).strip()
    if head != revision:
        raise RuntimeError("GitHub deployment commit mismatch")
    for filename in ("dgx_batch.py", "t5_t8_dgx.py"):
        blob = subprocess.check_output(["git", "-C", str(deployment), "show", f"{revision}:experiments/mom_bpe/{filename}"])
        path = directory / filename
        if path.is_symlink() or not path.is_file() or path.read_bytes() != blob:
            raise RuntimeError(f"Missing or inconsistent GitHub source {filename}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(python), "-I", "-B", str(directory / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--deployment-revision", revision,
                    "--session", session, *options], cwd=source, env=env, check=True)


backup_approved = False
checkpoint_backup_approved = False
repository = ""
if backup_approved:
    options = ["--hub-checkpoints"] if checkpoint_backup_approved else []
    run("backup-hub", "--enable", "--hub-repo", repository, "--scope", "all", *options)
else:
    print("Private backup disabled")


## 17. Write the final session report


In [ ]:
import os
import re
import subprocess
from pathlib import Path

root = Path("/jupyter/socse.user63/mom-bpe-private-recovery")
if root.resolve() != root or not root.is_dir() or root.stat().st_uid != os.getuid() or root.stat().st_mode & 0o077:
    raise RuntimeError("Expected the existing user-owned private recovery root")
python = Path("/jupyter/socse.user63/venvs/mom-bpe-dgx-py312/bin/python")
revision = os.environ.get("MOM_BPE_DEPLOY_REVISION", "")
if not re.fullmatch("[0-9a-f]{40}", revision):
    raise ValueError("Set MOM_BPE_DEPLOY_REVISION to the verified GitHub deployment commit")
operational = "9b872ffb5a66bdf4d23df58acebbba7e788915d8"
source = root / ("resource-source-" + operational)
deployment = root / ("batch-source-" + revision)
if source.resolve() != source or deployment.resolve() != deployment:
    raise RuntimeError("Unexpected source symlink")

def run(action, *options):
    directory = deployment / "experiments/mom_bpe"
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="-1", HF_HUB_DISABLE_TELEMETRY="1",
               HF_HUB_OFFLINE="0" if action in ("authenticate", "restore-individual", "backup-hub") else "1",
               HF_DATASETS_OFFLINE="1", TRANSFORMERS_OFFLINE="1")
    env.pop("MLTOK_PLUGINS", None)
    head = subprocess.check_output(["git", "-C", str(deployment), "rev-parse", "HEAD"], text=True).strip()
    if head != revision:
        raise RuntimeError("GitHub deployment commit mismatch")
    for filename in ("dgx_batch.py", "t5_t8_dgx.py"):
        blob = subprocess.check_output(["git", "-C", str(deployment), "show", f"{revision}:experiments/mom_bpe/{filename}"])
        path = directory / filename
        if path.is_symlink() or not path.is_file() or path.read_bytes() != blob:
            raise RuntimeError(f"Missing or inconsistent GitHub source {filename}")
    session = os.environ.get("MOM_BPE_BATCH_SESSION", "session-1")
    subprocess.run([str(python), "-I", "-B", str(directory / "dgx_batch.py"), action,
                    "--root", str(root), "--source", str(source), "--deployment-revision", revision,
                    "--session", session, *options], cwd=source, env=env, check=True)

run("report")
